# 02 — Contrats de données avec Pandera

**Projet** : Prédiction d'attrition client (churn télécom) avec scikit-learn
**Objectif** : transformer les règles métier découvertes en EDA en **contrats exécutables**.

Un contrat de données vaut par ses deux propriétés :

1. il **accepte** les données conformes (sinon il bloque la production pour rien) ;
2. il **refuse** les données corrompues avec un message exploitable (sinon il ne sert à rien).

Ce notebook démontre les deux — y compris en **provoquant volontairement** des échecs.

## Objectifs pédagogiques

1. Lire un schéma `DataFrameModel` comme une documentation (types, bornes, catégories, unicité).
1. Provoquer et interpréter un échec de validation (`failure_cases`).
1. Distinguer les trois contrats du cycle de vie : brut, transformé, inférence.
1. Utiliser le mode `lazy` pour remonter toutes les erreurs d'un coup.

**Objectifs transverses du dépôt**

- Composer un pipeline scikit-learn propre : ColumnTransformer, transformers custom, fit sur le train uniquement.
- Utiliser une classe abstraite BaseModel pour rendre le framework interchangeable.
- Lire des métriques de classification en contexte déséquilibré (ROC AUC, PR AUC, rappel, précision).

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402
from loguru import logger  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
logger.remove()
logger.add(sys.stderr, level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (1500 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 1500

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.7)")
print(f"Cible             : {CONFIG.data.target}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

Projet            : telecom-churn-sklearn
Tâche             : binary
Métrique primaire : roc_auc (seuil cible : 0.7)
Cible             : churned
Algorithme        : random_forest (Forêt aléatoire scikit-learn)
Lignes (notebook) : 1500


In [2]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

2026-09-13 05:46:57 | INFO     | src.data.generators:generate:174 - Generating 1500 customers | seed=42 segments=4 positive_rate=0.26


2026-09-13 05:46:57 | INFO     | src.data.generators:generate:210 - Dataset generated | rows=1500 cols=15 churn_rate=0.264 missing_cells=123


data/raw vide : génération synthétique en mémoire (`make data` la persiste)
shape = (1500, 15)


,customer_id,signup_date,tenure_months,contract_type,internet_service,payment_method,region,monthly_charges,total_charges,support_tickets_6m,avg_monthly_data_gb,num_products,has_promotion,satisfaction_score,churned
0,CUS-00001,2021-03-04,59,one_year,dsl,electronic_check,south,40.65,2434.25,0,242.68,1,0,7.74,0
1,CUS-00002,2025-04-17,9,two_year,fiber,electronic_check,east,77.24,680.86,1,86.06,1,0,6.64,1
2,CUS-00003,2020-02-07,72,two_year,fiber,electronic_check,east,56.22,4066.63,1,10.43,3,0,6.81,0
3,CUS-00004,2025-03-29,10,one_year,none,credit_card,west,23.57,239.34,1,1.46,2,0,7.14,0
4,CUS-00005,2025-12-05,1,one_year,dsl,bank_transfer,north,45.59,45.38,0,93.38,4,1,8.35,0


**Ce qu'il faut retenir**

- Le `RawDataLoader` applique déjà le contrat au chargement : une source déviante échoue **ici**, pas en entraînement.
- `validate=False` existe pour inspecter des données cassées sans exception (diagnostic).

## 1. Le contrat des données brutes

Trois schémas cohabitent dans `src/data/schemas.py` :

| Schéma | Appliqué sur | Politique |
| --- | --- | --- |
| `RawDataSchema` | `data/raw` juste après chargement | strict, types et bornes imposés |
| `ProcessedDataSchema` | matrice livrée au modèle | 100 % numérique, zéro NaN |
| `InferenceDataSchema` | toute requête de prédiction | colonnes optionnelles, nulls tolérés |

In [3]:
from src.data.schemas import RawDataSchema, schema_to_markdown

display(Markdown(schema_to_markdown("raw")))

### `RawDataSchema`

| Colonne | Type | Nullable | Requis | Unique | Checks |
| --- | --- | --- | --- | --- | --- |
| `customer_id` | string[pyarrow] | False | True | False | <Check str_matches: str_matches('^CUS-[0-9]{5}$')> |
| `signup_date` | datetime64[ns] | False | True | False | - |
| `tenure_months` | int64 | False | True | False | <Check greater_than_or_equal_to: greater_than_or_equal_to(0)>; <Check less_than_or_equal_to: less_than_or_equal_to(72)> |
| `contract_type` | string[pyarrow] | False | True | False | <Check isin: isin(['month_to_month', 'one_year', 'two_year'])> |
| `internet_service` | string[pyarrow] | False | True | False | <Check isin: isin(['fiber', 'dsl', 'none'])> |
| `payment_method` | string[pyarrow] | False | True | False | <Check isin: isin(['electronic_check', 'mailed_check', 'bank_transfer', 'credit_card'])> |
| `region` | string[pyarrow] | False | True | False | <Check isin: isin(['north', 'south', 'east', 'west'])> |
| `monthly_charges` | float64 | False | True | False | <Check greater_than_or_equal_to: greater_than_or_equal_to(18.0)>; <Check less_than_or_equal_to: less_than_or_equal_to(480.0)> |
| `total_charges` | float64 | False | True | False | <Check greater_than_or_equal_to: greater_than_or_equal_to(18.0)>; <Check less_than_or_equal_to: less_than_or_equal_to(20000.0)> |
| `support_tickets_6m` | int64 | False | True | False | <Check greater_than_or_equal_to: greater_than_or_equal_to(0)>; <Check less_than_or_equal_to: less_than_or_equal_to(14)> |
| `avg_monthly_data_gb` | float64 | True | True | False | <Check greater_than_or_equal_to: greater_than_or_equal_to(0.0)>; <Check less_than_or_equal_to: less_than_or_equal_to(1000.0)> |
| `num_products` | int64 | False | True | False | <Check greater_than_or_equal_to: greater_than_or_equal_to(1)>; <Check less_than_or_equal_to: less_than_or_equal_to(5)> |
| `has_promotion` | int64 | False | True | False | <Check isin: isin([0, 1])> |
| `satisfaction_score` | float64 | True | True | False | <Check greater_than_or_equal_to: greater_than_or_equal_to(1.0)>; <Check less_than_or_equal_to: less_than_or_equal_to(10.0)> |
| `churned` | int64 | False | True | False | <Check isin: isin([0, 1])> |

**Ce qu'il faut retenir**

- Cette table est **générée depuis le code** : elle ne peut pas diverger de l'implémentation.
- `nullable=False` sur la clé et `unique` garantissent l'intégrité du jeu (pas de doublon silencieux).
- Les `checks` (bornes, `isin`) sont la traduction directe des règles métier du README.

In [4]:
from src.data.schemas import describe_schema

describe_schema("raw")

,dtype,nullable,required,unique,checks
column,,,,,
customer_id,string[pyarrow],False,True,False,<Check str_matches: str_matches('^CUS-[0-9]{5}...
signup_date,datetime64[ns],False,True,False,
tenure_months,int64,False,True,False,<Check greater_than_or_equal_to: greater_than_...
contract_type,string[pyarrow],False,True,False,"<Check isin: isin(['month_to_month', 'one_year..."
internet_service,string[pyarrow],False,True,False,"<Check isin: isin(['fiber', 'dsl', 'none'])>"
payment_method,string[pyarrow],False,True,False,"<Check isin: isin(['electronic_check', 'mailed..."
region,string[pyarrow],False,True,False,"<Check isin: isin(['north', 'south', 'east', '..."
monthly_charges,float64,False,True,False,<Check greater_than_or_equal_to: greater_than_...
total_charges,float64,False,True,False,<Check greater_than_or_equal_to: greater_than_...


## 2. Validation nominale : le contrat doit passer

In [5]:
from src.data.schemas import validate_frame, validation_report

validated = validate_frame(raw, "raw")
profile = validation_report(validated)
print(f"validation OK | lignes={profile['n_rows']} | colonnes={profile['n_columns']}")
print(f"              | cellules manquantes={profile['missing_cells']}")
validated.head(3)

2026-09-13 05:46:57 | INFO     | src.data.schemas:validate_frame:242 - Validation 'RawDataSchema' succeeded | rows=1500 cols=15


validation OK | lignes=1500 | colonnes=15
              | cellules manquantes=123


,customer_id,signup_date,tenure_months,contract_type,internet_service,payment_method,region,monthly_charges,total_charges,support_tickets_6m,avg_monthly_data_gb,num_products,has_promotion,satisfaction_score,churned
0,CUS-00001,2021-03-04,59,one_year,dsl,electronic_check,south,40.65,2434.25,0,242.68,1,0,7.74,0
1,CUS-00002,2025-04-17,9,two_year,fiber,electronic_check,east,77.24,680.86,1,86.06,1,0,6.64,1
2,CUS-00003,2020-02-07,72,two_year,fiber,electronic_check,east,56.22,4066.63,1,10.43,3,0,6.81,0


**Ce qu'il faut retenir**

- La coercition (`coerce=True`) absorbe les différences Parquet/CSV : un entier lu comme flottant reste valide.
- Un contrat qui ne passe **jamais** en local est un contrat mal calibré — le vérifier fait partie du travail.

## 3. Échecs volontaires — la partie la plus utile du notebook

On corrompt **délibérément** le dataset pour vérifier que le contrat mord. Chaque cellule
isole une violation ; l'exception est capturée puis affichée avec ses `failure_cases`.

In [6]:
try:
    import pandera.pandas as pa
except ModuleNotFoundError:  # pandera < 0.26
    import pandera as pa

SchemaViolation = (pa.errors.SchemaError, pa.errors.SchemaErrors)


def show_violation(label: str, frame: pd.DataFrame) -> None:
    """Validate a deliberately corrupted frame and explain the failure.

    Args:
        label: Human readable name of the injected corruption.
        frame: Corrupted dataset.
    """
    try:
        RawDataSchema.validate(frame, lazy=True)
    except SchemaViolation as error:
        cases = getattr(error, "failure_cases", None)
        print(f"[REFUSÉ] {label}")
        if cases is not None:
            print(cases.head(6).to_string(index=False))
        else:
            print(error)
        return
    print(f"[ACCEPTÉ — ATTENTION] {label} : le contrat ne couvre pas ce cas")

In [7]:
numeric_bounded = [
    name
    for name, column in RawDataSchema.to_schema().columns.items()
    if any(
        getattr(check, "name", "") in {"ge", "greater_than_or_equal_to", "in_range"}
        for check in (getattr(column, "checks", []) or [])
    )
]
column = numeric_bounded[0]
corrupted = raw.copy()
corrupted.loc[corrupted.index[:5], column] = 10_000_000
show_violation(f"valeur hors bornes sur `{column}` (10 000 000)", corrupted)

[REFUSÉ] valeur hors bornes sur `tenure_months` (10 000 000)
schema_context        column                     check  check_number  failure_case  index
        Column tenure_months less_than_or_equal_to(72)             1      10000000      0
        Column tenure_months less_than_or_equal_to(72)             1      10000000      1
        Column tenure_months less_than_or_equal_to(72)             1      10000000      2
        Column tenure_months less_than_or_equal_to(72)             1      10000000      3
        Column tenure_months less_than_or_equal_to(72)             1      10000000      4


**Ce qu'il faut retenir**

- `failure_cases` donne la **colonne**, le **check** et les **valeurs** en échec : le diagnostic est immédiat.
- En production, cette erreur doit faire échouer le run (fail fast) plutôt que d'entraîner un modèle sur des données fausses.

In [8]:
corrupted = raw.copy()
corrupted["colonne_non_declaree"] = 0
show_violation("colonne non déclarée (strict=True)", corrupted)

[REFUSÉ] colonne non déclarée (strict=True)
       column         failure_case index  schema_context            check check_number
RawDataSchema colonne_non_declaree  None DataFrameSchema column_in_schema         None


In [9]:
corrupted = raw.drop(columns=[raw.columns[-1]])
show_violation("colonne manquante", corrupted)

[REFUSÉ] colonne manquante
       column failure_case index  schema_context               check check_number
RawDataSchema      churned  None DataFrameSchema column_in_dataframe         None


In [10]:
categorical_checked = [
    name
    for name, column in RawDataSchema.to_schema().columns.items()
    if any(getattr(check, "name", "") == "isin" for check in (getattr(column, "checks", []) or []))
]
if categorical_checked:
    column = categorical_checked[0]
    corrupted = raw.copy()
    corrupted.loc[corrupted.index[:3], column] = "modalite_inexistante"
    show_violation(f"catégorie hors liste sur `{column}`", corrupted)
else:
    print("Aucune colonne contrainte par `isin` dans ce schéma.")

[REFUSÉ] catégorie hors liste sur `contract_type`
schema_context        column                                            check  check_number         failure_case  index
        Column contract_type isin(['month_to_month', 'one_year', 'two_year'])             0 modalite_inexistante      0
        Column contract_type isin(['month_to_month', 'one_year', 'two_year'])             0 modalite_inexistante      1
        Column contract_type isin(['month_to_month', 'one_year', 'two_year'])             0 modalite_inexistante      2


**Ce qu'il faut retenir**

- Une nouvelle modalité non déclarée est le bug silencieux le plus fréquent après un changement de SI amont.
- Deux réponses possibles : mettre à jour le contrat (évolution légitime) ou refuser (régression).

In [11]:
corrupted = raw.copy()
corrupted.loc[corrupted.index[1], CONFIG.data.id_column] = corrupted.loc[
    corrupted.index[0], CONFIG.data.id_column
]
show_violation(f"clé dupliquée sur `{CONFIG.data.id_column}`", corrupted)

[REFUSÉ] clé dupliquée sur `customer_id`
 schema_context      column                      check check_number failure_case  index
DataFrameSchema customer_id multiple_fields_uniqueness         None    CUS-00001      0
DataFrameSchema customer_id multiple_fields_uniqueness         None    CUS-00001      1


In [12]:
corrupted = raw.copy()
corrupted.loc[corrupted.index[0], CONFIG.data.id_column] = None
show_violation(f"clé nulle sur `{CONFIG.data.id_column}`", corrupted)

[REFUSÉ] clé nulle sur `customer_id`
schema_context      column        check check_number failure_case  index
        Column customer_id not_nullable         None          NaN      0


**Ce qu'il faut retenir**

- Une clé dupliquée crée une **fuite** entre splits : la même observation peut se retrouver en train et en test.
- C'est pourquoi `assert_no_overlap()` est testé dans `tests/test_loaders.py`.

## 4. Mode `lazy` : tout remonter d'un coup

In [13]:
column = numeric_bounded[0]
corrupted = raw.copy()
corrupted.loc[corrupted.index[:20], column] = -1_000_000
corrupted.loc[corrupted.index[20:40], column] = 1_000_000
show_violation(f"40 violations sur `{column}` (mode lazy)", corrupted)

[REFUSÉ] 40 violations sur `tenure_months` (mode lazy)
schema_context        column                       check  check_number  failure_case  index
        Column tenure_months greater_than_or_equal_to(0)             0      -1000000      0
        Column tenure_months greater_than_or_equal_to(0)             0      -1000000      1
        Column tenure_months greater_than_or_equal_to(0)             0      -1000000      2
        Column tenure_months greater_than_or_equal_to(0)             0      -1000000      3
        Column tenure_months greater_than_or_equal_to(0)             0      -1000000      4
        Column tenure_months greater_than_or_equal_to(0)             0      -1000000      5


**Ce qu'il faut retenir**

- Sans `lazy`, pandera s'arrête à la première erreur : on découvre les problèmes un par un.
- Avec `lazy`, le rapport complet permet de corriger le flux amont en une fois (configurable via `data.validation.lazy`).

## 5. Contrat des données transformées

In [14]:
from src.data.schemas import ProcessedDataSchema

try:
    ProcessedDataSchema.validate(raw)
except SchemaViolation as error:
    print("[REFUSÉ comme attendu] la matrice brute n'est pas numérique :")
    print(str(error)[:320])

numeric_matrix = raw.select_dtypes(include=[np.number]).dropna().head(50)
print("matrice numérique valide ->", ProcessedDataSchema.validate(numeric_matrix).shape)

[REFUSÉ comme attendu] la matrice brute n'est pas numérique :
DataFrameSchema 'ProcessedDataSchema' failed series or dataframe validator 2: <Check all_numeric>
matrice numérique valide -> (50, 9)


**Ce qu'il faut retenir**

- Ce contrat est la **dernière ligne de défense** avant le modèle : aucune feature texte, aucun NaN, aucun infini.
- Il est appliqué automatiquement par `TrainPipeline` quand `data.validation.processed: true`.

## 6. Contrat d'inférence : tolérant mais pas laxiste

In [15]:
from src.data.schemas import InferenceDataSchema

payload = (
    raw.drop(columns=[CONFIG.data.target]).head(20).copy()
    if CONFIG.data.target
    else raw.head(20).copy()
)
payload.iloc[0, 0] = None  # valeur manquante tolérée
payload["colonne_du_client"] = "web"  # colonne supplémentaire tolérée
print("payload accepté ->", InferenceDataSchema.validate(payload).shape)

bad_payload = payload.copy()
if numeric_bounded:
    bad_payload[numeric_bounded[0]] = "pas-un-nombre"
try:
    InferenceDataSchema.validate(bad_payload)
except SchemaViolation as error:
    print("[REFUSÉ] type incohérent :", str(error)[:200])

payload accepté -> (20, 15)


[REFUSÉ] type incohérent : {
    "DATA": {
        "DATATYPE_COERCION": [
            {
                "schema": "InferenceDataSchema",
                "column": "tenure_months",
                "check": "coerce_dtype('Int64')


**Ce qu'il faut retenir**

- La cible est absente d'un payload d'inférence : le contrat ne doit pas l'exiger (`required=False`).
- Tolérer les nulls et les colonnes en trop **sans** renoncer aux checks de type : c'est l'équilibre recherché.
- Une requête partielle est corrigée par le preprocessing ; une requête incohérente est refusée avec un message clair.

## 7. Où la validation s'exécute vraiment

| Emplacement | Contrat | Déclencheur |
| --- | --- | --- |
| `RawDataLoader.load()` | `RawDataSchema` | chaque chargement de données |
| `TrainPipeline._preprocess()` | `ProcessedDataSchema` | avant entraînement |
| `Predictor.predict()` | `InferenceDataSchema` | chaque requête de prédiction |
| `tests/test_data_schemas.py` | les trois | chaque commit (`make test`) |

### Checklist à reproduire sur un nouveau projet

1. Écrire le schéma **avant** le code de chargement (le contrat guide l'implémentation).
2. Ajouter un test d'acceptation (données conformes) **et** un test de refus (données corrompues).
3. Versionner le schéma avec le code : une évolution de contrat est un changement d'API.
4. Journaliser les `failure_cases` : ce sont eux qui font gagner du temps en incident.